# 📥 Étape 1 — Collecte & Extraction Multi-Sources en Arbres (`Parent -> Enfant`) 💻 *(Exécution Locale)*

**Environnement** : Ce notebook tourne **100% en local sur ton PC (CPU)** pour ne gaspiller aucune minute de GPU Colab.

**Objectif pédagogique** : Apprendre à structurer une collecte de données pour un LLM conversationnel en séparant dès l'extraction :
1. **Les textes isolés** (pour le *Continued Pre-training* — CPT).
2. **Les fils de discussion `Parent -> Enfant`** (pour le *Supervised Fine-Tuning* — SFT).

> 📖 **Conseil** : Lis d'abord `01_data_collection/README.md` et assure-toi d'avoir installé `pip install -r ../requirements.txt`.

In [3]:
import os
import re
import json
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Tuple
from dotenv import load_dotenv

# Chargement des variables d'environnement depuis le fichier .env à la racine du dépôt
PROJECT_ROOT = Path("..").resolve()
load_dotenv(PROJECT_ROOT / ".env")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Racine du projet : {PROJECT_ROOT}")
print(f"✅ Fichier WhatsApp ../_chat.txt présent : {(PROJECT_ROOT / '_chat.txt').exists()}")

Note: you may need to restart the kernel to use updated packages.
✅ Racine du projet : C:\Users\willi\AppData\Local\Programs
✅ Fichier WhatsApp ../_chat.txt présent : False



[notice] A new release of pip is available: 23.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## 1. Capitaliser sur l'Existant : Amélioration du Parser WhatsApp (`Cleansing_Dataset.py`)

Dans ton fichier existant `../Cleansing_Dataset.py`, tu as déjà écrit `clean_chat_text()` et `parse_ios_whatsapp()`. 
Nous reprenons ta fonction `clean_chat_text()` ci-dessous, puis dans le **`TODO 1`**, tu vas améliorer l'extraction de `../_chat.txt` pour :
- Calculer l'écart de temps $\Delta t$ entre deux prises de parole.
- Ne créer un lien `parent_id` (dialogue SFT) **que si la réponse intervient dans un délai raisonnable** (ex: $\Delta t \le 30$ minutes). Sinon, le message devient un commentaire isolé (`parent_id = None`) pour le CPT !

In [4]:
# === CODE DÉJÀ RÉALISÉ DANS TON PROJET (Cleansing_Dataset.py) ===
def clean_chat_text(chat_text: str) -> Optional[str]:
    """Nettoyage de base repris de Cleansing_Dataset.py"""
    chat_text = str(chat_text)
    ignoring_tags = [
        "<Media omitted>", "You deleted this message", "This message was deleted",
        "Messages and calls are end-to-end encrypted", "<This message was edited>",
        "created group", "added you", "removed you", "audio omitted",
        "sticker omitted", "image omitted", "GIF omitted", "document omitted",
        "video omitted", "//", "voice call", "Voice call"
    ]
    for ignoring_tag in ignoring_tags:
        if ignoring_tag in chat_text:
            return None

    chat_text = re.sub(r'http\S+|www\.\S+', '', chat_text)  # Suppression URLs
    chat_text = re.sub(r'@\s*[^\s@]+', '', chat_text)       # Suppression mentions (@)
    chat_text = re.sub(r'(.)\1{2,}', r'\1\1', chat_text)    # Réduction répétitions (mdrrr -> mdrr)
    chat_text = re.sub(r'\s+', ' ', chat_text).strip()
    return chat_text if chat_text else None

In [15]:
# === EXERCICE GUIDÉ : TODO 1 ===
def parse_whatsapp_to_records(lines: List[str], max_reply_delay_minutes: float = 30.0) -> List[Dict]:
    """
    Transforme les lignes d'un export WhatsApp iOS (_chat.txt) en enregistrements standardisés
    avec 'comment_id' et 'parent_id' basés sur la proximité temporelle.
    """
    regex_ios = r'^\[(\d{2}/\d{2}/\d{2,4},\s+\d{2}:\d{2}:\d{2})\] ([^:]+): (.+)$'
    grouped_turns = []  # Liste de dicts: {'speaker': ..., 'timestamp': datetime, 'text': ...}
    
    current_speaker = None
    current_time = None
    current_buffer = []

    for line in lines:
        line = line.replace('\u200e', '').replace('\u2069', '').strip()
        match = re.match(regex_ios, line)
        if match:
            date_str, speaker, message = match.groups()
            clean_msg = clean_chat_text(message)
            if not clean_msg:
                continue
            
            date_clean = re.sub(r'\s+', ' ', date_str)
            fmt = "%d/%m/%Y, %H:%M:%S" if len(date_clean.split(',')[0].split('/')[-1]) == 4 else "%d/%m/%y, %H:%M:%S"
            msg_time = datetime.strptime(date_clean, fmt)

            if speaker == current_speaker:
                current_buffer.append(clean_msg)
                current_time = msg_time
            else:
                if current_speaker and current_buffer:
                    full_text = " ".join(current_buffer)
                    if len(full_text.split()) >= 2:
                        grouped_turns.append({
                            "speaker": current_speaker,
                            "timestamp": current_time,
                            "text": full_text
                        })
                current_speaker = speaker
                current_time = msg_time
                current_buffer = [clean_msg]
        else:
            if current_speaker:
                clean_line = clean_chat_text(line)
                if clean_line:
                    current_buffer.append(clean_line)

    if current_speaker and current_buffer:
        full_text = " ".join(current_buffer)
        if len(full_text.split()) >= 2:
            grouped_turns.append({
                "speaker": current_speaker,
                "timestamp": current_time,
                "text": full_text
            })

    records = []
    for i, turn in enumerate(grouped_turns):
        comment_id = f"wa_{i}"
        parent_id = None
        
        # TODO 1 : Calcule la différence de temps en minutes avec `grouped_turns[i-1]` si i > 0.
        # Astuce : delta_minutes = (turn["timestamp"] - grouped_turns[i-1]["timestamp"]).total_seconds() / 60.0
        # Si 0 <= delta_minutes <= max_reply_delay_minutes, affecte `parent_id = f"wa_{i-1}"`
        
        # --- TON CODE ICI ---
        if i > 0 :
            delta_minutes = (turn["timestamp"]-grouped_turns[i-1]["timestamp"]).total_seconds() / 60.0
            if 0 <= delta_minutes <= max_reply_delay_minutes :
                parent_id = f"wa_{i-1}"
            

        records.append({
            "comment_id": comment_id,
            "parent_id": parent_id,
            "text": turn["text"],
            "like_count": 0,
            "source_video_title": "WhatsApp Chat",
            "source_platform": "whatsapp"
        })
    return records

In [16]:
# 🧪 Cellule de validation du TODO 1
sample_whatsapp_lines = [
    "[25/12/2025, 14:00:10] Alice: Mince perr, Eneo a encore fait son travail ?",
    "[25/12/2025, 14:02:30] Bob: Gars c'est la magie, on va dormir dans le noir ce soir.",
    "[26/12/2025, 09:15:00] Alice: Les gars qui a le lien de la réunion Outlook ?"
]
test_wa_records = parse_whatsapp_to_records(sample_whatsapp_lines, max_reply_delay_minutes=30.0)
assert len(test_wa_records) == 3, "Il devrait y avoir 3 messages extraits."
assert test_wa_records[0]["parent_id"] is None, "Le 1er message ne doit pas avoir de parent."
assert test_wa_records[1]["parent_id"] == "wa_0", "Le 2e message (2 min plus tard) doit avoir 'wa_0' comme parent_id !"
assert test_wa_records[2]["parent_id"] is None, "Le 3e message (le lendemain) ne doit PAS être lié à wa_1 !"
print("🎉 Bravo ! Ton parser WhatsApp avec fenêtre temporelle fonctionne parfaitement.")

# Test direct sur ton vrai fichier ../_chat.txt local
chat_file = PROJECT_ROOT / "_chat.txt"
if chat_file.exists():
    with open(chat_file, "r", encoding="utf-8") as f:
        real_wa_records = parse_whatsapp_to_records(f.readlines(), max_reply_delay_minutes=30.0)
    print(f"📱 Extraction depuis ../_chat.txt : {len(real_wa_records)} tours de parole extraits en local.")

🎉 Bravo ! Ton parser WhatsApp avec fenêtre temporelle fonctionne parfaitement.


---
## 2. Collecte YouTube Data API v3 en Arbres (`commentThreads.list`) (`TODO 2`)

Pour récupérer automatiquement des milliers de commentaires camerounais depuis ton PC :
1. Crée une clé API gratuite sur [Google Cloud Console](https://console.cloud.google.com/) en activant **YouTube Data API v3**.
2. Place-la dans ton fichier local `../.env` sous le nom `YOUTUBE_API_KEY="..."`.
3. Complète le **`TODO 2`** ci-dessous pour transformer la réponse JSON de l'API YouTube en notre format standard (`comment_id`, `parent_id`, `text`, `like_count`, `source_video_title`).

In [21]:
# === EXERCICE GUIDÉ : TODO 2 ===
def parse_youtube_thread_item(item: Dict, video_title: str) -> List[Dict]:
    """
    Prend un élément 'item' renvoyé par youtube.commentThreads().list(part='snippet,replies')
    et retourne une liste contenant :
      - Le commentaire principal (parent_id = None)
      - Toutes ses réponses éventuelles (parent_id = ID du commentaire principal)
    """
    extracted = []

    # TODO 2.1 : Extraire le commentaire principal (topLevelComment)
    # Astuce : récupère `top_comment = item["snippet"]["topLevelComment"]`
    # Son ID est `top_comment["id"]`, son texte est `top_comment["snippet"]["textOriginal"]`
    # et ses likes sont `top_comment["snippet"].get("likeCount", 0)`.
    
    top_comment = item["snippet"]["topLevelComment"]
    parent_id = top_comment["id"]
    top_snippet = top_comment["snippet"]
    extracted.append({
      "comment_id": top_comment["id"],
      "parent_id": None,
      "text": top_snippet["textOriginal"],
      "like_count": top_snippet.get("likeCount",0),
      "source_video_title": video_title,
      "source_platform": "youtube"
    })


    # TODO 2.2 : Vérifier si la clé "replies" est présente dans `item`.
    # Si oui, parcourir `item["replies"]["comments"]` et ajouter chaque réponse dans `extracted`
    # avec `"parent_id": parent_id` (l'ID du topLevelComment).
    if "replies" in item :
      for reply in item["replies"]["comments"] :
        reply_snippet = reply["snippet"]
        extracted.append({
          "comment_id": reply["id"],
          "parent_id": parent_id,
          "text": reply_snippet["textOriginal"],
          "like_count": reply_snippet.get("likeCount",0),
          "source_video_title": video_title,
          "source_platform": "youtube"
        }) 

    return extracted

In [22]:
# 🧪 Cellule de validation du TODO 2
mock_yt_item = {
    "id": "UgxThread123",
    "snippet": {
        "topLevelComment": {
            "id": "UgxParent001",
            "snippet": {
                "textOriginal": "Pourquoi l'artiste n'a pas sorti le son hier soir ?",
                "likeCount": 25
            }
        }
    },
    "replies": {
        "comments": [
            {
                "id": "UgxChild002",
                "snippet": {
                    "parentId": "UgxParent001",
                    "textOriginal": "Gars le mbom a ndem, il a préféré go gérer ses ngata au quartier !",
                    "likeCount": 8
                }
            }
        ]
    }
}

parsed_yt = parse_youtube_thread_item(mock_yt_item, video_title="Debat Musique Urbaine 237")
assert len(parsed_yt) == 2, "Tu dois extraire 2 commentaires (1 parent + 1 enfant)."
assert parsed_yt[0]["comment_id"] == "UgxParent001" and parsed_yt[0]["parent_id"] is None
assert parsed_yt[1]["comment_id"] == "UgxChild002" and parsed_yt[1]["parent_id"] == "UgxParent001"
print("🎉 Bravo ! L'extracteur d'arbres YouTube fonctionne comme attendu.")

🎉 Bravo ! L'extracteur d'arbres YouTube fonctionne comme attendu.


---
## 3. Scraper Local avec Pagination & Sauvegarde Incrémentale (`TODO 3`)

Complète le **`TODO 3`** pour écrire au fil de l'eau sur ton disque dur dans `../data/raw/raw_comments.jsonl`.

In [28]:
# === EXERCICE GUIDÉ : TODO 3 ===
def fetch_youtube_video_comments(
    youtube_client,
    video_id: str,
    video_title: str,
    output_jsonl_path: Path = RAW_DATA_DIR / "raw_comments.jsonl",
    max_pages: int = 5
) -> int:
    """
    Récupère les pages de commentaires d'une vidéo YouTube et les ajoute (append)
    au fichier JSONL local au fur et à mesure.
    """
    saved_count = 0
    next_page_token = None

    for page in range(max_pages):
        request = youtube_client.commentThreads().list(
            part="snippet,replies",
            videoId=video_id,
            maxResults=100,
            pageToken=next_page_token,
            textFormat="plainText"
        )
        response = request.execute()

        # TODO 3 : Pour chaque `item` dans `response.get("items", [])` :
        # 1. Appelle `parse_youtube_thread_item(item, video_title)`
        # 2. Écris chaque commentaire extrait en une ligne JSON dans `output_jsonl_path` (mode 'a', encoding='utf-8')
        # 3. Incrémente `saved_count`
        
        with open(output_jsonl_path, "a", encoding="utf-8") as output_file:
            for item in response.get("items",[]) :
                comments = parse_youtube_thread_item(item, video_title)
                for comment in comments :
                    output_file.write(json.dumps(comment))
                    saved_count +=1


        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    return saved_count

---
## 4. Séparation des Commentaires Isolés (CPT) et des Paires `Parent -> Enfant` (SFT) (`TODO 4`)

Une fois ton fichier local `../data/raw/raw_comments.jsonl` alimenté (avec WhatsApp et/ou YouTube), on sépare les arbres `Parent -> Enfant` des commentaires isolés.

In [39]:
# === EXERCICE GUIDÉ : TODO 4 ===
def split_trees_and_isolated(records: List[Dict]) -> Tuple[List[Dict], List[Dict]]:
    """
    Prend une liste d'enregistrements bruts (avec 'comment_id' et 'parent_id')
    et retourne deux listes :
      1. `sft_pairs` : liste de dicts {"parent_id": ..., "child_id": ..., "user": texte_parent, "assistant": texte_enfant, "source": ...}
      2. `isolated_comments` : liste des enregistrements sans parent_id qui n'ont reçu aucune réponse.
    """
    by_id = {r["comment_id"]: r for r in records}
    parents_with_replies = set()
    sft_pairs = []

    # TODO 4.1 : Parcourir `records`. Si `r["parent_id"]` n'est pas None et existe dans `by_id` :
    #   - Récupérer le commentaire parent via `parent_rec = by_id[r["parent_id"]]`
    #   - Ajouter `r["parent_id"]` dans le set `parents_with_replies`
    #   - Ajouter la paire dans `sft_pairs`
    
    for r in records :
      if r["parent_id"] is not None and r["parent_id"] in by_id :
        parent_rec = by_id[r["parent_id"]]
        parents_with_replies.add(r["parent_id"])
        sft_pairs.append({"parent_id": r["parent_id"],"child_id": r["comment_id"],"user": parent_rec["text"],"assistant": r["text"],"source": r.get("source_video_title")})
        


    # TODO 4.2 : Construire `isolated_comments` contenant tous les `r` de `records`
    # tels que `r["parent_id"] is None` ET `r["comment_id"] not in parents_with_replies`.
    isolated_comments = []
    
    for r in records :
      if r["parent_id"] is None and r["comment_id"] not in parents_with_replies :
        isolated_comments.append(r)
    
    # --- TON CODE ICI ---

    # --------------------

    return sft_pairs, isolated_comments

In [40]:
# 🧪 Cellule de validation du TODO 4
sample_records = [
    {"comment_id": "c1", "parent_id": None, "text": "Pourquoi il n'a pas sorti le son ?", "source_video_title": "Clip A"},
    {"comment_id": "c2", "parent_id": "c1", "text": "Gars le mbom a ndem au studio hier !", "source_video_title": "Clip A"},
    {"comment_id": "c3", "parent_id": None, "text": "Ce son me rappelle trop le kwatt à Douala quand il pleut fort.", "source_video_title": "Clip A"}
]

pairs, isolated = split_trees_and_isolated(sample_records)
assert len(pairs) == 1 and pairs[0]["user"] == "Pourquoi il n'a pas sorti le son ?"
assert len(isolated) == 1 and isolated[0]["comment_id"] == "c3"
print("🎉 Étape 1 locale validée ! Tu peux maintenant ouvrir 02_filtering_scoring/02_Filtering_and_Scoring.ipynb en local.")

🎉 Étape 1 locale validée ! Tu peux maintenant ouvrir 02_filtering_scoring/02_Filtering_and_Scoring.ipynb en local.
